<a href="https://colab.research.google.com/github/kosebaris1/MLP_Flask/blob/main/MLP_Flask_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



### Öğrenci Bilgileri
* **Adınız:** Barış
* **Soyadınız:** Köse
* **Okul Numaranız:** 2212721022
* **GitHub Repo Bağlantısı:** https://github.com/kosebaris1/MLP_Flask




#  Balık Ağırlık Tahmin Projesi

## Proje Amacı
Bu projede, balığın fiziksel özelliklerini (uzunluk, yükseklik, genişlik, tür) kullanarak makine öğrenmesi ile balığın ağırlığını tahmin eden bir regresyon modeli geliştirilecektir.

## Veri Seti
- **Kaynak**: Kaggle - Fish Market Dataset
- **Hedef Değişken**: Weight (Ağırlık - gram)
- **Özellikler**: Length1, Length2, Length3, Height, Width, Species

## 1. Veri Yükleme ve Keşifsel Veri Analizi

Bu bölümde veri seti yüklenip temel istatistiksel bilgiler incelenecektir.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Fish.csv")
df.head()


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


### 1.1. Veri Seti İnceleme

Aşağıdaki kod ile veri setinin yapısı, eksik veri durumu ve temel istatistikler incelenecektir.

- `df.info()`: Veri setinin genel yapısı (satır sayısı, kolonlar, veri tipleri)
- `df.describe()`: Sayısal değişkenler için temel istatistikler (ortalama, standart sapma, min, max vb.)
- `df.isnull().sum()`: Her kolondaki eksik veri sayısı

In [ ]:
df.info()
df.describe()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Species  159 non-null    object 
 1   Weight   159 non-null    float64
 2   Length1  159 non-null    float64
 3   Length2  159 non-null    float64
 4   Length3  159 non-null    float64
 5   Height   159 non-null    float64
 6   Width    159 non-null    float64
dtypes: float64(6), object(1)
memory usage: 8.8+ KB


,0
Species,0
Weight,0
Length1,0
Length2,0
Length3,0
Height,0
Width,0


## 2. Veri Ön İşleme (Data Preprocessing)

### 2.1. Eksik Veri Analizi
Veri setinde eksik veri olup olmadığı kontrol edilecektir. Eğer eksik veri varsa, uygun yöntemle (ortalama, medyan, mod vb.) doldurulacak veya gerekçelendirilerek silinecektir.

**Gerekçe**: Eksik veriler modelin performansını düşürebilir. Bu yüzden öncelikle eksik veri analizi yapılmalıdır.

### 2.2. Kategorik Veri Kodlama

**Kullanılan Yöntem**: One-Hot Encoding (get_dummies)

**Gerekçe**:
- Species (Balık Türü) kategorik bir değişkendir ve sayısal değerlere dönüştürülmesi gerekir
- One-Hot Encoding kullanılmasının nedeni: Türler arasında sıralı bir ilişki yoktur (örn: Bream > Pike gibi bir sıralama yoktur)
- `drop_first=True` parametresi ile multicollinearity (çoklu doğrusal bağlantı) problemi önlenir
- Bu sayede n-1 adet dummy değişken oluşturulur (n = tür sayısı)

In [ ]:
df_encoded = pd.get_dummies(df, columns=["Species"], drop_first=True)
df_encoded.head()


,Weight,Length1,Length2,Length3,Height,Width,Species_Parkki,Species_Perch,Species_Pike,Species_Roach,Species_Smelt,Species_Whitefish
0,242.0,23.2,25.4,30.0,11.5200,4.0200,False,False,False,False,False,False
1,290.0,24.0,26.3,31.2,12.4800,4.3056,False,False,False,False,False,False
2,340.0,23.9,26.5,31.1,12.3778,4.6961,False,False,False,False,False,False
3,363.0,26.3,29.0,33.5,12.7300,4.4555,False,False,False,False,False,False
4,430.0,26.5,29.0,34.0,12.4440,5.1340,False,False,False,False,False,False


### 2.3. Özellik ve Hedef Değişken Ayrımı

Model eğitimi için özellikler (X) ve hedef değişken (y) ayrılacaktır.

- **X (Özellikler)**: Weight hariç tüm kolonlar
- **y (Hedef)**: Weight (tahmin edilecek değişken)

In [ ]:
X = df_encoded.drop("Weight", axis=1)
y = df_encoded["Weight"]


### 2.4. Veri Bölünmesi (Train-Test Split)

**Yöntem**: Train-Test Split
- **Test Oranı**: %20 (test_size=0.2)
- **Random State**: 42 (tekrarlanabilirlik için)

**Gerekçe**: Modelin genelleme yeteneğini test etmek için veri seti eğitim ve test olarak ikiye ayrılır.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


### 2.5. Veri Ölçekleme (Normalizasyon)

**Kullanılan Yöntem**: StandardScaler

**Gerekçe**:
- Length1, Length2, Length3, Height, Width gibi özellikler farklı ölçeklerdedir (cm cinsinden)
- Ölçekleme yapılmazsa, büyük değerlere sahip özellikler (örn: Length3) model üzerinde daha fazla etkili olabilir
- StandardScaler kullanarak tüm özellikler ortalaması 0, standart sapması 1 olacak şekilde normalize edilir
- Bu işlem modelin daha iyi performans göstermesini sağlar

**Not**: Scaler sadece eğitim verisiyle fit edilir, test verisi sadece transform edilir (data leakage önlemek için)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 3. Backward Elimination (Geriye Doğru Eleme)

### 3.1. Backward Elimination Yöntemi

**Amaç**: En anlamlı özellikleri seçerek modeli sadeleştirmek ve overfitting'i önlemek

**Yöntem**:
1. Tüm özelliklerle başlangıç modeli kurulur
2. Her özelliğin p-value değeri kontrol edilir
3. En yüksek p-value'ya sahip özellik (p > 0.05 ise) çıkarılır
4. Model tekrar eğitilir
5. Tüm özelliklerin p-value'su < 0.05 olana kadar işlem tekrarlanır

**Gerekçe**:
- p-value > 0.05 olan özellikler istatistiksel olarak anlamsızdır
- Gereksiz özelliklerin çıkarılması modelin daha yorumlanabilir olmasını sağlar
- Model karmaşıklığı azalır ve genelleme yeteneği artar

### 3.2. Veri Hazırlama

Backward Elimination için veriler hazırlanacak ve sabit terim (intercept) eklenecektir.

In [ ]:
X_train_be = pd.DataFrame(X_train_scaled, columns=X_train.columns)
y_train_be = y_train.reset_index(drop=True)

# Indexleri sıfırla
X_train_be = X_train_be.reset_index(drop=True)


In [ ]:
import statsmodels.api as sm

X_be = sm.add_constant(X_train_be)


### 3.3. Backward Elimination Süreci

Her iterasyonda:
1. Model eğitilir
2. En yüksek p-value bulunur
3. Eğer p-value > 0.05 ise, o özellik çıkarılır
4. İşlem tüm özellikler anlamlı olana kadar devam eder

**Seçilen Özellikler**: Backward Elimination sonrası kalan özellikler model için en önemli özelliklerdir.

In [ ]:
cols = list(X_be.columns)

while True:
    model = sm.OLS(y_train_be, X_be[cols]).fit()
    p_values = model.pvalues

    pmax = p_values.max()
    feature_with_pmax = p_values.idxmax()

    print("\nEn yüksek p-value:", feature_with_pmax, "=", pmax)

    # Eğer 0.05'ten büyükse çıkartıyoruz
    if pmax > 0.05:
        print("Çıkarılıyor →", feature_with_pmax)
        cols.remove(feature_with_pmax)
    else:
        print("Tüm değişkenler anlamlı. BE tamamlandı.")
        break

print("\nSon seçilen değişkenler:", cols)

model_be = sm.OLS(y_train_be, X_be[cols]).fit()
model_be.summary()



En yüksek p-value: Width = 0.7981368527802571
Çıkarılıyor → Width

En yüksek p-value: Species_Roach = 0.5712629496414136
Çıkarılıyor → Species_Roach

En yüksek p-value: Species_Whitefish = 0.7053326343361945
Çıkarılıyor → Species_Whitefish

En yüksek p-value: Species_Perch = 0.5328451875693693
Çıkarılıyor → Species_Perch

En yüksek p-value: Length3 = 0.34267886184990326
Çıkarılıyor → Length3

En yüksek p-value: Height = 0.6674679006539526
Çıkarılıyor → Height

En yüksek p-value: Length1 = 0.1578311583889584
Çıkarılıyor → Length1

En yüksek p-value: Species_Parkki = 0.04605531381021552
Tüm değişkenler anlamlı. BE tamamlandı.

Son seçilen değişkenler: ['const', 'Length2', 'Species_Parkki', 'Species_Pike', 'Species_Smelt']


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Weight   R-squared:                       0.926
Model:                            OLS   Adj. R-squared:                  0.924
Method:                 Least Squares   F-statistic:                     382.5
Date:                Tue, 16 Dec 2025   Prob (F-statistic):           5.33e-68
Time:                        16:49:10   Log-Likelihood:                -758.92
No. Observations:                 127   AIC:                             1528.
Df Residuals:                     122   BIC:                             1542.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const            386.7945      8.626     44.841      0.000     369.719     403.870
Length2          419.1617     12.298     34.083      0.000     394.816     443.507
Species_Parkki    18.2247      9.043      2.015      0.046       0.324      36.125
Species_Pike    -113.4562     10.832    -10.474      0.000    -134.900     -92.012
Species_Smelt     62.7466      9.905      6.335      0.000      43.138      82.355
==============================================================================
Omnibus:                       28.940   Durbin-Watson:                   1.917
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               51.689
Skew:                           1.018   Prob(JB):                     5.97e-12
Kurtosis:                       5.370   Cond. No.                         2.45
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### 3.4. Final Model Özeti

Yukarıdaki model özetinde:
- **R² (R-squared)**: Modelin veriyi açıklama yüzdesi (0.926 = %92.6)
- **p-value**: Her özelliğin istatistiksel anlamlılığı (p < 0.05 anlamlı)
- **Coefficients**: Her özelliğin ağırlık katsayıları

**Seçilen Özellikler**: const, Length2, Species_Parkki, Species_Pike, Species_Smelt

## 4. Model Değerlendirme

### 4.1. Test Seti Hazırlama

Test seti de aynı şekilde ölçeklenip, Backward Elimination sonrası seçilen özelliklerle hazırlanacaktır.

In [ ]:
# Test setini aynı şekilde DataFrame yapıyoruz
X_test_be = pd.DataFrame(X_test_scaled, columns=X_test.columns)
X_test_be = sm.add_constant(X_test_be)

# Sadece BE’de kalan kolonları alıyoruz
X_test_final = X_test_be[cols]

y_pred = model_be.predict(X_test_final)


### 4.2. Model Performans Metrikleri

**Kullanılan Metrikler**:

1. **R² (R-squared)**: Modelin veriyi açıklama yüzdesi
   - 0 ile 1 arasında değer alır
   - 1'e yakın olması modelin iyi olduğunu gösterir

2. **MAE (Mean Absolute Error)**: Ortalama mutlak hata
   - Gerçek ve tahmin edilen değerler arasındaki farkın ortalaması
   - Düşük olması tercih edilir (gram cinsinden)

3. **MSE (Mean Squared Error)**: Ortalama karesel hata
   - Hataların karesinin ortalaması
   - Büyük hataları daha fazla cezalandırır

4. **RMSE (Root Mean Squared Error)**: Karekök ortalama karesel hata
   - MSE'nin karekökü
   - Orijinal birimle aynı ölçekte olduğu için yorumlanması kolaydır (gram cinsinden)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("R²:", r2)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)


R²: 0.9477461652957282
MAE: 65.42586383387584
MSE: 7432.548182778588
RMSE: 86.21222757114323


In [ ]:
import pickle

with open("fish_model.pkl", "wb") as f:
    pickle.dump(model_be, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)


## 5. Model ve Preprocessing Nesnelerinin Kaydedilmesi

Eğitilen model ve scaler nesneleri pickle formatında kaydedilecektir. Bu sayede Flask uygulamasında kullanılmak üzere hazır hale getirilecektir.

**Kaydedilen Dosyalar**:
- `fish_model.pkl`: Eğitilmiş regresyon modeli
- `scaler.pkl`: Veri ölçekleme için kullanılan StandardScaler nesnesi
- `feature_names.json`: Tüm özellik isimleri (scaler ile uyumluluk için)
- `be_cols.json`: Backward Elimination sonrası seçilen özellikler

In [ ]:
import json

# 1) Eğitimde kullanılan tüm feature isimleri
feature_names = list(X.columns)

with open("feature_names.json", "w") as f:
    json.dump(feature_names, f)

# 2) BE sonrası seçilen kolonlar
with open("be_cols.json", "w") as f:
    json.dump(cols, f)

from google.colab import files
files.download("feature_names.json")
files.download("be_cols.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Sonuç ve Özet

### 6.1. Seçilen Özellikler
Backward Elimination sonrası seçilen özellikler:
- **Length2**: Balığın çapraz uzunluğu (en önemli özellik)
- **Species_Parkki**: Parkki türü için dummy değişken
- **Species_Pike**: Pike türü için dummy değişken
- **Species_Smelt**: Smelt türü için dummy değişken

### 6.2. Model Performansı
- **R² Score**: ~0.95 (Model veriyi %95 oranında açıklamaktadır)
- **MAE**: ~65 gram (Ortalama tahmin hatası)
- **RMSE**: ~86 gram (Kök ortalama karesel hata)

### 6.3. Yorumlar
- Model başarılı bir şekilde balık ağırlığını tahmin edebilmektedir
- Length2 özelliği en önemli tahmin edici değişkendir
- Bazı balık türleri (Parkki, Pike, Smelt) ağırlık tahmininde anlamlıdır
- Model Flask uygulamasında kullanılmak üzere hazırdır